###Spark Aggregate Functions

####Funciones simples de agregacion

In [0]:
%run "../includes/configuration"

In [0]:
movies_df  = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
from pyspark.sql.functions import count, countDistinct, sum, max, avg, min

In [0]:
movies_df.select(count("*")).show()

+--------+
|count(1)|
+--------+
|    4725|
+--------+



In [0]:
movies_df.count()

4725

In [0]:
movies_df.select(count("year_release_date")).show()

+------------------------+
|count(year_release_date)|
+------------------------+
|                    4426|
+------------------------+



In [0]:
movies_df.select(countDistinct("year_release_date")).show()

+---------------------------------+
|count(DISTINCT year_release_date)|
+---------------------------------+
|                               90|
+---------------------------------+



In [0]:
movies_df.select(sum("budget")).display()

sum(budget)
1.40606455289E11


In [0]:
movies_df.filter("year_release_date = 2016") \
    .select(sum("budget"), count("movie_id")) \
        .withColumnRenamed("sum(budget)", "total_budget") \
            .withColumnRenamed("count(movie_id)", "count_movies") \
        .display()

total_budget,count_movies
4.29015E9,94


In [0]:
display(movies_df)

movie_id,title,budget,popularity,year_release_date,release_date,revenue,duration_time,vote_average,vote_count,ingestion_date,enviroment
5,Four Rooms,4000000.0,22.87623,1995,1995-12-09,4300000.0,98,6.5,530,2026-09-21T09:32:22.401419Z,PRODUCTION
11,Star Wars,1.1E7,126.393695,1977,1977-05-25,7.75398007E8,121,8.1,6624,2026-09-21T09:32:22.401419Z,PRODUCTION
12,Finding Nemo,9.4E7,85.688789,2003,2003-05-30,9.40335536E8,100,7.6,6122,2026-09-21T09:32:22.401419Z,PRODUCTION
13,Forrest Gump,5.5E7,138.133331,1994,1994-07-06,6.77945399E8,142,8.2,7927,2026-09-21T09:32:22.401419Z,PRODUCTION
14,American Beauty,1.5E7,80.878605,1999,1999-09-15,3.56296601E8,122,7.9,3313,2026-09-21T09:32:22.401419Z,PRODUCTION
16,Dancer in the Dark,1.28E7,22.022228,2000,2000-05-17,4.0031879E7,140,7.6,377,2026-09-21T09:32:22.401419Z,PRODUCTION
18,The Fifth Element,9.0E7,109.528572,1997,1997-05-07,2.6392018E8,126,7.3,3885,2026-09-21T09:32:22.401419Z,PRODUCTION
19,Metropolis,9.262E7,32.351527,1927,1927-01-10,650422.0,153,8.0,657,2026-09-21T09:32:22.401419Z,PRODUCTION
20,My Life Without Me,2000000.0,7.958831,2003,2003-03-07,2600000.0,106,7.2,77,2026-09-21T09:32:22.401419Z,PRODUCTION
22,Pirates of the Caribbean: The Curse of the Black Pearl,1.4E8,271.972889,2003,2003-07-09,6.55011224E8,143,7.5,6985,2026-09-21T09:32:22.401419Z,PRODUCTION


####Group By

In [0]:
movie_group_by_df = movies_df \
    .groupBy("year_release_date") \
        .agg(
            sum("budget").alias("total_budget"),
            avg("budget").alias("avg_budget"),
            max("budget").alias("max_budget"),
            min("budget").alias("min_budget"),
            count("movie_id").alias("count_movies")
        )

In [0]:
display(movie_group_by_df)

year_release_date,total_budget,avg_budget,max_budget,min_budget,count_movies
1959,1.1383848E7,3794616.0,5000000.0,2883848.0,3
1990,6.56225E8,2.6249E7,7.0E7,225000.0,25
1975,2.67E7,4450000.0,1.1E7,300000.0,6
1977,1.3611E8,9074000.0,2.2E7,10000.0,15
2003,5.173137898E9,3.359180453246753E7,2.0E8,46000.0,154
2007,5.71114751E9,3.1379931373626374E7,3.0E8,15000.0,182
1974,3.3085E7,4135625.0,1.3E7,85000.0,8
1927,9.262E7,9.262E7,9.262E7,9.262E7,1
1955,3200000.0,1600000.0,2000000.0,1200000.0,2
2006,6.398129867E9,2.9484469433179725E7,2.7E8,100000.0,217


####Window Functions

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc, dense_rank

In [0]:
movie_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))
movie_dense_rank = Window.partitionBy("year_release_date").orderBy(desc("budget"))

movies_df.select("title", "budget", "year_release_date") \
    .filter("year_release_date is not null") \
    .withColumn("rank", rank().over(movie_rank)) \
    .withColumn("dense_rank", dense_rank().over(movie_dense_rank)) \
    .display()

title,budget,year_release_date,rank,dense_rank
Intolerance,385907.0,1916,1,1
The Big Parade,245000.0,1925,1,1
Metropolis,9.262E7,1927,1,1
Die Büchse der Pandora,2000000.0,1929,1,1
The Broadway Melody,379000.0,1929,2,2
Hell's Angels,3950000.0,1930,1,1
A Farewell to Arms,2000000.0,1932,1,1
42nd Street,439000.0,1933,1,1
It Happened One Night,325000.0,1934,1,1
Top Hat,609000.0,1935,1,1
